In [8]:
import os
import glob
import pandas as pd
import numpy as np

ptdfs_path = r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\PTDFs"
limits_path = r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\Limits"



ptdf_dfs = {}
limits_dfs = {}



In [41]:
def load_csv_folder(folder_path, sep=";", decimal="."):
    """
    Load all CSV files from a folder into a dictionary of DataFrames.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing CSV files.
    sep : str, optional
        CSV separator, by default ";"
    decimal : str, optional
        Decimal separator, by default "."

    Returns
    -------
    dict
        Dictionary where:
        - key = file name without extension
        - value = pandas DataFrame
    """
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    dfs = {}

    for file in csv_files:
        name = os.path.splitext(os.path.basename(file))[0]
        df = pd.read_csv(file, sep=sep, decimal=decimal)
        dfs[name] = df

    return dfs

def filter_ptdf_columns(df):
    """
    This function receives a ptdf result dataframe from powerfactory and clean the columns
    It returns a cleaned dataframe with only the ptdf for lines and trafos of the 
    original dataframe 
    """
    df.rename(columns={df.columns[0]:"Injection Bus"}, inplace=True)

    first_col = df.columns[0] # Keep the bus index
    cols = [first_col]
    seen_elements = set()
    df = df.iloc[1:].reset_index(drop = True)
    for c in df.columns[1:]:
        # keep only lines and trafos
        if not (c.startswith("Line") or c.startswith("Trf")):
            continue

        # remove .1, .2 etc
        if any(f".{i}" in c for i in range(1, 10)):
            continue

        # avoid duplicates (just in case)
        if c not in seen_elements:
            cols.append(c)
            seen_elements.add(c)

    return df[cols]

def reshape_ptdf(df):
    """
    Transform PTDF dataframe from wide to long format.

    Parameters
    ----------
    df : pandas.DataFrame
        PTDF dataframe (wide format)

    Returns
    -------
    pandas.DataFrame
        Long format with columns:
        - Injection Bus
        - Element
        - PTDF
    """
    df_long = df.melt(
        id_vars="Injection Bus",
        var_name="Element",
        value_name="PTDF"
    )

    df_long["PTDF"] = pd.to_numeric(df_long["PTDF"],errors = "coerce")
    df_long["PTDF"] = df_long["PTDF"].fillna(0)

    return df_long

def combine_limits(limits_dfs):
    """
    Combine line and transformer limits into a single dataframe.

    Parameters
    ----------
    limits_dfs : dict
        Dictionary containing limits dataframes
        Expected keys: 'limits_lines', 'limits_trafos'

    Returns
    -------
    pandas.DataFrame
        Combined limits dataframe with unified 'Element' column
    """
    # Copy to avoid modifying original data
    limit_lines = limits_dfs["limits_lines"].copy()
    limit_trafos = limits_dfs["limits_trafos"].copy()

    # Standardize column name
    limit_lines = limit_lines.rename(columns={"name": "Element"})
    limit_trafos = limit_trafos.rename(columns={"name": "Element"})

    # Combine
    limit_elements = pd.concat([limit_lines, limit_trafos], ignore_index=True)

    return limit_elements

def merge_ptdf_with_limits(df_long, limits_df):
    """
    Merge one PTDF long dataframe with the combined limits dataframe.

    Parameters
    ----------
    df_long : pandas.DataFrame
        Long PTDF dataframe with columns like:
        - Injection Bus
        - Element
        - PTDF
    limits_df : pandas.DataFrame
        Combined limits dataframe with column:
        - Element

    Returns
    -------
    pandas.DataFrame
        Merged and sorted dataframe
    """
    result = df_long.merge(limits_df, on="Element", how="left")

    result["Injection Bus"] = pd.to_numeric(
        result["Injection Bus"], errors="coerce"
    )
    result["Injection Bus"] = result["Injection Bus"].astype("Int64")

    result = result.sort_values(by=["Injection Bus", "Element"])
    result = result.reset_index(drop=True)

    return result

def merge_ptdf_with_limits_dict(ptdf_dict,loading_dict):
    results = {}
    for case in ptdf_dict.keys():
        df_ptdf = ptdf_dict[case]
        df_loading = loading_dict[case]

        df = df_ptdf.merge(df_loading,on = "Element", how = "left")
        df["Injection Bus"] = pd.to_numeric(df["Injection Bus"], errors="coerce").astype("Int64")

        df = df.sort_values(by=["Injection Bus", "Element"]).reset_index(drop=True)

        results[case] = df

    return results






def calculate_transfer_limit(df):
    """
    Calculate transfer limit based on PTDF and element limits.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing:
        - PTDF
        - smax
        - p_used

    Returns
    -------
    pandas.DataFrame
        DataFrame with added 'Transfer Limit' column
    """
    # Calculate transfer limit
    margin = df["smax"] - df["p_used"]
    df["Transfer Limit"] = np.where(
        df["PTDF"] == 0,
        np.inf,  # element not limiting
        (margin/ df["PTDF"]).abs())

    return df


In [42]:
ptdf_dfs = load_csv_folder(ptdfs_path)
limits_dfs = load_csv_folder(limits_path)

ptdf_dfs_filtered = {}
for name,df in ptdf_dfs.items():
    filtered_df = filter_ptdf_columns(df)
    filtered_df = filtered_df.reset_index(drop = True)
    ptdf_dfs_filtered[name] = filtered_df

ptdf_long_dfs = {}
for name, df in ptdf_dfs_filtered.items():
    df = reshape_ptdf(df)
    ptdf_long_dfs[name] = df


limits_combined = combine_limits(limits_dfs)

results = {}
for name,df_long in ptdf_long_dfs.items():
    result = merge_ptdf_with_limits(df_long,limits_combined)
    results[name] = result

for name,result in results.items():
    df = calculate_transfer_limit(result)
    results[name] = df

for name,result in results.items():
    groups = result.groupby("Injection Bus")
    print(f"\n CASE: {name}")
    for bus,df_bus in groups:
        top5 = df_bus.sort_values("Transfer Limit").head(10)
        print(f"\nBus {bus}")
        print(top5[["Element", "PTDF", "Transfer Limit"]])




 CASE: Distribution_Factors_Results_(SYM)_Base

Bus 1
         Element      PTDF  Transfer Limit
35   Trf 06 - 31  1.005349      142.177731
8   Line 05 - 06  0.537879      264.227273
2   Line 02 - 03  0.446103      520.982676
10  Line 06 - 07 -0.257771      651.812421
0   Line 01 - 02  0.549658      858.324198
1   Line 01 - 39  0.450342      976.890920
13  Line 08 - 09 -0.448873     1090.120640
11  Line 06 - 11 -0.208733     1126.074831
4   Line 03 - 04  0.376103     1168.763316
14  Line 09 - 39 -0.449094     1173.092938

Bus 2
         Element      PTDF  Transfer Limit
81   Trf 06 - 31  0.998067      143.215074
54  Line 05 - 06  0.558934      254.273853
48  Line 02 - 03  0.633141      367.077688
50  Line 03 - 04  0.534940      821.728398
57  Line 06 - 11 -0.284849      825.170450
61  Line 10 - 11  0.261502      881.262759
52  Line 04 - 05  0.497678      925.988540
65  Line 15 - 16 -0.248558     1019.358163
56  Line 06 - 07 -0.153282     1096.138748
63  Line 13 - 14 -0.286378     1133

# Logic of the code

1. Read the CSV results from Powerfactory into a dictionary of dataframes : raw_dfs_ptdfs
2. Read the CSV limit results from Powerfactory : raw_dfs_limts
3. Filter the columns of the raw Dataframe of PTDFS to extract only the columns of interest : dfs_ptdfs_filtered
4. Combine the limits into one single Dataframe of limits: dfs_limits_combined
5. Reshape the dfs_ptdfs_filtered dictionary, melt function, transform from wide to long format, id: injection bus, variable: Element, value: PTDF (create function)
6. Rename the dfs_limits_combined dataframe column: "name" to "Element" so merging is possible
7. Merge on element, how: left -> as a result the dictionary of results is created 
8. Transform the relevant columns into the correct type of variable so operating is possible, do it  for each df in the dict of dfs
9. Calculate the transfer limit of each element, for each bus, for each case
10. Order the results to observe the max to the less impacted elements for each bus/case
11. Present the table results

In [47]:
import re
import pandas as pd

def normalize(key):
    key = key.replace("Distribution_Factors_Results_(SYM)_","")
    key = key.replace("Base Case", "Base")
    key = re.sub(r"_+", "_", key)
    return key


def calculate_transfer_limit(df, eps=1e-6):
    df = df.copy()

    margin = df["smax"] - df["actual"]

    df["Transfer Limit"] = np.where(
        margin.notna() & df["PTDF"].notna() & (df["PTDF"] > eps),
        margin / df["PTDF"],
        np.inf
    )

    return df

def calculate_transfer_limit_2(df):
    df = df.copy()

    margin = df["smax"] - df["actual"]

    df["Transfer Limit"] = np.where(
        df["PTDF"] == 0,
        np.inf,  # element not limiting
        (margin / df["PTDF"]).abs()
    )

    return df
# PTDFS
ptdf_dfs = load_csv_folder(ptdfs_path)
ptdf_dfs_filtered = {}
for name, df in ptdf_dfs.items():
    filtered_df = filter_ptdf_columns(df)
    filtered_df = filtered_df.reset_index(drop=True)
    ptdf_dfs_filtered[name] = filtered_df

ptdf_long_dfs = {}
for name, df in ptdf_dfs_filtered.items():
    df_long = reshape_ptdf(df)
    ptdf_long_dfs[name] = df_long

# LOADINGS FROM CSV
df_loadings = pd.read_csv(r"C:\Users\UI450907\Desktop\TE RWEST\Tesis\Phase3Results\loadings_all.csv")
loading_dict = {}

for case,df_case in df_loadings.groupby("case"):
    df_case = df_case.copy()
    df_case = df_case.rename(columns={"name":"Element"})
    loading_dict[case] = df_case

# build normalized mapping

ptdf_map = {normalize(k): k for k in ptdf_long_dfs.keys()}
loading_map = {normalize(k): k for k in loading_dict.keys()}

valid_cases = set(ptdf_map.keys()) & set(loading_map.keys())

#Merge by case
results = {}

for case in valid_cases:
    ptdf_name = ptdf_map[case]
    loading_name = loading_map[case]

    df_long = ptdf_long_dfs[ptdf_name]
    df_loading = loading_dict[loading_name]

    result = merge_ptdf_with_limits(df_long, df_loading)
    results[case] = result

for name, result in results.items():
    results[name] = calculate_transfer_limit_2(result)


In [48]:
for name, result in results.items():
    groups = result.groupby("Injection Bus")
    print(f"\nCASE: {name}")
    for bus, df_bus in groups:
        top10 = df_bus.sort_values("Transfer Limit").head(10)
        print(f"\nBus {bus}")
        print(top10[["Element", "PTDF", "Transfer Limit"]])



CASE: Line_01_39_Cnt

Bus 1
         Element      PTDF  Transfer Limit
35   Trf 06 - 31  0.991486      140.787046
2   Line 02 - 03  0.806360      171.642259
8   Line 05 - 06  0.577832      265.998347
0   Line 01 - 02  1.000000      523.536232
11  Line 06 - 11 -0.353083      549.777464
4   Line 03 - 04  0.679884      558.822688
15  Line 10 - 11  0.324469      596.006513
19  Line 15 - 16 -0.313826      698.594366
6   Line 04 - 05  0.636729      852.530384
17  Line 13 - 14 -0.355420     1026.319844

Bus 2
         Element      PTDF  Transfer Limit
81   Trf 06 - 31  0.991496      140.785626
48  Line 02 - 03  0.806401      171.633532
54  Line 05 - 06  0.577836      265.996505
57  Line 06 - 11 -0.353087      549.771236
50  Line 03 - 04  0.679894      558.814469
61  Line 10 - 11  0.324474      595.997329
65  Line 15 - 16 -0.313825      698.596592
52  Line 04 - 05  0.636733      852.525029
63  Line 13 - 14 -0.355425     1026.305407
62  Line 10 - 13 -0.324474     1078.849157

Bus 3
          E

# HCA Analysis Powerfactory
The results coming from here are generated with actual HCA analysis from powerfactory
